# Bandcamp Artist Discog Scraper
Based on:
- dbeley's [bandcamp-library-scraper.py](https://github.com/dbeley/bandcamp-library-scraper/blob/main/bandcamp-library-scraper.py)
- nahnhh's [webcrawl2_movie details.ipynb](https://github.com/nahnhh/top-movies-2005-2025/blob/main/webcrawl2_movie%20details.ipynb)

Artist list used: `slushwave-bandcamp-links.txt` compiled in Slushwave Social Club Discord server.

`image-links.csv` only gets the smallest file size, the one ending with [_1x1_700.avif](https://f4.bcbits.com/img/a3705632330_1x1_700.avif)

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
BASE_DIR = Path.cwd()
LINKS_FILE = BASE_DIR / "slushwave-bandcamp-links.txt"
OUTPUT_FILE = BASE_DIR / "bc-albums-image-links.csv"
IMAGES_DIR = BASE_DIR / "images"

### List of browers to rotate, spin them once a while

In [20]:
HEADERS_LIST = [
  {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0",
    "X-Requested-With": "XMLHttpRequest"
  },
  {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.13; rv:110.0) Gecko/20100101 Firefox/110.0",
    "X-Requested-With": "XMLHttpRequest"
  },
  {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest"
  },
  {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36 Edg/110.0.1587.50",
    "X-Requested-With": "XMLHttpRequest"
  },
  {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest"
  }
]

In [ ]:
def parse_album_page(url, soup):
  """Fetch an album page and extract year, tracklist, and runtime."""

  # Release year
  year = None
  try:
    # Try datePublished meta tag first
    meta_date = soup.find("meta", {"itemprop": "datePublished"})
    if meta_date and meta_date.get("content"):
      year = meta_date.get("content")[:4]
    else:
      # Try to extract from description meta tag (e.g., "released 28 May 2026")
      meta_desc = soup.find("meta", {"name": "description"})
      if meta_desc and meta_desc.get("content"):
        desc_text = meta_desc.get("content")
        m = re.search(r"released.*?(\d{4})", desc_text, re.IGNORECASE)
        if m:
          year = m.group(1)
        else:
          # Fallback: search for any 4-digit year in description
          m = re.search(r"(19|20)\d{2}", desc_text)
          if m:
            year = m.group(0)
      else:
        # Final fallback: search for a 4-digit year in credits or release text
        credits = soup.select_one(".tralbum-credits, .credits, .release-date")
        if credits:
          m = re.search(r"(19|20)\d{2}", credits.get_text())
          if m:
            year = m.group(0)
  except Exception:
    year = None

  # Tracklist and runtime
  tracklist = []
  total_seconds = 0
  # bandcamp uses table#track_table or ol.track_list / ol#track_table
  track_rows = soup.select('table#track_table tr.track_row, ol#track_table li, ol.track_list li, li.track')
  if not track_rows:
    track_rows = soup.select('div.trackList > div.track')
  for tr in track_rows:
    # title
    title = None
    t = tr.find(class_='title') or tr.find('span', {'itemprop': 'name'}) or tr.find('a')
    if t:
      title = t.get_text(strip=True)
    else:
      title = tr.get_text(strip=True)
    # duration
    dur = None
    time_tag = tr.find(class_='time') or tr.find('span', {'class': 'time'})
    if time_tag:
      dur = time_tag.get_text(strip=True)
    else:
      m = re.search(r"(\d{1,2}:\d{2}(?::\d{2})?)", tr.get_text())
      if m:
        dur = m.group(1)
    secs = 0
    if dur:
      parts = dur.split(':')
      try:
        if len(parts) == 2:
          secs = int(parts[0]) * 60 + int(parts[1])
        elif len(parts) == 3:
          secs = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
      except Exception:
        secs = 0
    total_seconds += secs
    tracklist.append({"title": title, "duration": dur})

  runtime = None
  if total_seconds:
    m, s = divmod(total_seconds, 60)
    runtime = f"{m}:{s:02d}"

### Async webscraping

#### Scraping bandcamp functions

In [22]:
import requests
from bs4 import BeautifulSoup

def parse_artist_name(raw_text: str) -> str:
    """Trim the 'by ' prefix used by Bandcamp artist labels without chopping the name."""
    text = raw_text.strip()
    if text.lower().startswith("by "):
        return text[3:].strip()
    if text.lower().startswith("par "):
        return text[4:].strip()
    return text

def big_image(image_link: str) -> str:
    """Convert a Bandcamp image link to the biggest version."""
    if image_link.endswith("_1x1_700.avif"):
        return image_link[:-13] + "_0"
    return image_link

def smol_image(image_link: str) -> str:
    """Convert a Bandcamp image link to the smallest version."""
    if image_link.endswith("_16.jpg"):
        return image_link[:-6] + "_1x1_700.avif"
    return image_link

In [ ]:
import csv
import logging
import re
from datetime import datetime

def get_soup(url: str, session: requests.Session, headers: dict) -> BeautifulSoup:
	response = session.get(url, headers=headers, timeout=30)
	response.raise_for_status()
	return BeautifulSoup(response.content, "lxml")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()


def extract_image_url(album_element) -> str:
	"""Extract the image URL from album element."""
	try:
		# Find the picture element within the album item
		picture = album_element.find("picture")
		if picture:
			# Look for source tags with the AVIF format
			source = picture.find("source")
			if source and source.get("srcset"):
				# srcset can contain multiple URLs, get the first one
				image_url = source.get("srcset").split()[0]
				return image_url
			# Fallback to img tag
			img = picture.find("img")
			if img and img.get("src"):
				return img.get("src")
	except Exception as e:
		print(f"Error extracting image URL: {e}")
	return "Error"


def download_image(session: requests.Session, image_url: str, filename: str) -> bool:
    """Download an image file from URL."""
    try:
        response = session.get(image_url, timeout=30)
        response.raise_for_status()
        with open(filename, "wb") as f:
            f.write(response.content)
        return True
    except Exception as e:
        print(f"Error downloading image from {image_url}: {e}")
    return False

def parse_artist_page(artist, session: requests.Session, headers: dict):
	soup_artist = get_soup(artist["url"] + "music", session, headers=headers)
	list_albums = []
	try:
		for item in (
			soup_artist.find("div", {"class": "leftMiddleColumns"})
			.find("ol", {"id": "music-grid"})
			.find_all("li", {"class": "music-grid-item"})
		):
			album_element = item.find("p", {"class": "title"})
			album_name = album_element.text.strip()
			alternate_artist = None
			if alternate_element := album_element.find(
				"span", {"class": "artist-override"}
			):
				logger.debug(
						f"Trying to parse alternate artist name in {album_element}"
				)
				album_name = album_name.split("\n")[0]
				alternate_artist = alternate_element.text.strip()
				logger.debug(f"Found {alternate_artist}")
			url = item.find("a")["href"]
			
			# Extract and download image
			image_url = extract_image_url(item)
			image_filename = None
			if image_url:
				image_filename = f"{artist['name']}_{album_name.replace('/', '_')}.avif"
				image_path = IMAGES_DIR / image_filename
				IMAGES_DIR.mkdir(exist_ok=True)
				if download_image(session, big_image(image_url), image_path):
					image_filename = image_filename
			
			list_albums.append(
					{
						"artist": artist["name"],
						"alias": alternate_artist,
						"album": album_name,
						"url": url if url.startswith("https://") else artist["url"] + url,
						"image": image_filename,
					}
			)
	except Exception as e:
		logger.warning(
			f"Couldn't extract informations for artist {artist['name']}: {e}"
		)
		return []
	
	return list_albums

def parse_album_page():
	pass

def export_to_csv(list_items, filename: str):
    with open(filename, "w", encoding="utf8", newline="") as output_file:
        fc = csv.DictWriter(
            output_file, fieldnames=sorted(set().union(*(d.keys() for d in list_items)))
        )
        fc.writeheader()
        fc.writerows(list_items)

In [ ]:
import json

def extract_jsonld(soup) -> dict:
    """Extract JSON-LD structured data from the page."""
    try:
        script = soup.find("script", {"type": "application/ld+json"})
        if script:
            jsonld_data = json.loads(script.string)
            return jsonld_data
    except Exception as e:
        logger.warning(f"Failed to extract JSON-LD: {e}")
    return {}

# Example usage:
# jsonld = extract_jsonld(soup)
# print(jsonld.get("name"))  # Album name
# print(jsonld.get("datePublished"))  # Release date
# print(jsonld.get("albumRelease"))  # Track information
# print(jsonld.get("byArtist"))  # Artist info

In [ ]:
def parse_jsonld_album(jsonld_data: dict) -> dict:
    """Extract album metadata from JSON-LD data."""
    result = {
        "name": jsonld_data.get("name"),
        "artist": None,
        "date_modified": jsonld_data.get("dateModified"),
        "image": None,
        "description": None,
        "price": None,
        "currency": None,
    }
    
    # Extract artist name
    by_artist = jsonld_data.get("byArtist", {})
    if isinstance(by_artist, dict):
        result["artist"] = by_artist.get("name")
    
    # Extract first album release details
    album_releases = jsonld_data.get("albumRelease", [])
    if album_releases and len(album_releases) > 0:
        first_release = album_releases[0]
        result["description"] = first_release.get("description")
        result["image"] = first_release.get("image", [None])[0] if first_release.get("image") else None
        
        offers = first_release.get("offers", {})
        result["price"] = offers.get("price")
        result["currency"] = offers.get("priceCurrency")
    
    return result

# Example: jsonld = extract_jsonld(soup)
#          album_info = parse_jsonld_album(jsonld)
#          print(album_info)

#### Async batches

In [ ]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from tqdm.asyncio import tqdm

max_concurrency = 10
sem = asyncio.Semaphore(max_concurrency)
timeout_urls = []

async def async_get_soup(url: str, session, header) -> BeautifulSoup:
	"""Fetch URL using aiohttp and return BeautifulSoup object (lxml)."""
	async with sem:
		try:
			async with session.get(url, header=header, timeout=30) as r:
				if r.status != 200:
					raise aiohttp.ClientResponseError(r.request_info, r.history, status=r.status)
				html = await r.text()
				return BeautifulSoup(html, "lxml")
			
		except asyncio.TimeoutError:
				logger.debug(f"async_get_soup timeout error for {url}")
				raise
		except Exception as e:
				logger.debug(f"async_get_soup failed for {url}: {e}")
				raise
			

async def parse_album_page_async(album_url: str, session: aiohttp.ClientSession) -> dict:
	"""Async parse an album page for year, tracklist, runtime, and label link."""
	try:
		soup = await async_get_soup(album_url, session, header=header)
	except Exception as e:
		logger.debug(f"Failed to fetch album page {album_url}: {e}")
		return {"year": None, "tracklist": [], "runtime": None, "label_url": None}

	year = None
	try:
		meta_date = soup.find("meta", {"itemprop": "datePublished"})
		if meta_date and meta_date.get("content"):
			year = meta_date.get("content")[:4]
		else:
			credits = soup.select_one(".tralbum-credits, .credits, .release-date")
			if credits:
				m = re.search(r"(19|20)\d{2}", credits.get_text())
				if m:
					year = m.group(0)
	except Exception:
		year = None

	tracklist = []
	total_seconds = 0
	track_rows = soup.select('table#track_table tr.track_row, ol#track_table li, ol.track_list li, li.track')
	if not track_rows:
		track_rows = soup.select('div.trackList > div.track')
	for tr in track_rows:
		title = None
		t = tr.find(class_='title') or tr.find('span', {'itemprop': 'name'}) or tr.find('a')
		if t:
			title = t.get_text(strip=True)
		else:
			title = tr.get_text(strip=True)

		dur = None
		time_tag = tr.find(class_='time') or tr.find('span', {'class': 'time'})
		if time_tag:
			dur = time_tag.get_text(strip=True)
		else:
			m = re.search(r"(\d{1,2}:\d{2}(?::\d{2})?)", tr.get_text())
			if m:
				dur = m.group(1)

		secs = 0
		if dur:
			parts = dur.split(':')
			try:
				if len(parts) == 2:
					secs = int(parts[0]) * 60 + int(parts[1])
				elif len(parts) == 3:
					secs = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
			except Exception:
				secs = 0
		total_seconds += secs
		tracklist.append({"title": title, "duration": dur})

	runtime = None
	if total_seconds:
		m, s = divmod(total_seconds, 60)
		runtime = f"{m}:{s:02d}"

	label_url = None
	for a in soup.select(".tralbum-credits a, .credits a, a"):
		href = a.get("href", "")
		text = a.get_text("").lower()
		if href and ("/label/" in href or "label" in text or "labels/" in href):
			label_url = href if href.startswith("http") else href
			break

	return {"year": year, "tracklist": tracklist, "runtime": runtime, "label_url": label_url}

async def parse_music_page_async(artist_url: str, session: aiohttp.ClientSession) -> list:
	"""Parse an artist Music page (async) and return basic album info list."""
	soup_artist = await async_get_soup(artist_url.rstrip('/') + "/music", session, header=header)
	albums = []
	try:
		grid = soup_artist.find("div", {"class": "leftMiddleColumns"})
		ol = grid.find("ol", {"id": "music-grid"}) if grid else None
		items = ol.find_all("li", {"class": "music-grid-item"}) if ol else []
		for item in items:
			album_element = item.find("p", {"class": "title"})
			album_name = album_element.text.strip() if album_element else None
			alternate_artist = None
			if album_element and (alternate_element := album_element.find("span", {"class": "artist-override"})):
				album_name = album_name.split("\n")[0]
				alternate_artist = alternate_element.text.strip()
			url = item.find("a")["href"] if item.find("a") else None
			url = url if url and url.startswith("http") else (artist_url.rstrip('/') + url if url else None)
			image_url = extract_image_url(item)
			image_filename = None
			if image_url:
				image_filename = f"{artist_url.split('/')[-1]}_{album_name.replace('/', '_')}.avif"
			albums.append({"artist": artist_url.split('/')[-1], "alias": alternate_artist, "album": album_name, "url": url, "image": image_filename})
	except Exception as e:
		logger.warning(f"Couldn't parse music page {artist_url}: {e}")
	return albums

async def extract_discography_async(artist, session: aiohttp.ClientSession) -> list:
	"""Async extract discography with album detail enrichment."""
	albums = await parse_music_page_async(artist["url"], session)
	if not albums:
		return []

	tasks = []
	for album in albums:
		if album.get("url"):
			tasks.append(parse_album_page_async(album["url"], session))
		else:
			tasks.append(asyncio.sleep(0, result={"year": None, "tracklist": [], "runtime": None, "label_url": None}))

	details_list = await asyncio.gather(*tasks, return_exceptions=True)
	for album, details in zip(albums, details_list):
		if isinstance(details, Exception):
			logger.warning(f"Failed to parse album details for {album.get('url')}: {details}")
			details = {"year": None, "tracklist": [], "runtime": None, "label_url": None}
		album.update(details)

	return albums

async def scrape_album_and_details(session, header, album_url):
	async with sem:
		try:
			details = await parse_album_page_async(album_url, session)
			return details
		except Exception as e:
			print(f"Error scraping album {album_url}: {e}")
			return None

async def scrape_batch(session, header, urls):
	"""Async scraping helper that can run album-detail tasks."""
	if session is None:
		async with aiohttp.ClientSession() as session:
			tasks = [scrape_album_and_details(session, header, url) for url in urls]
			return await tqdm.gather(*tasks)
	else:
		tasks = [scrape_album_and_details(session, header, url) for url in urls]
		return await tqdm.gather(*tasks)


#### Run it!

In [25]:
extract_discography_async(artist = {"url": "https://nagligijaqarniq.bandcamp.com/"})

TypeError: extract_discography_async() missing 1 required positional argument: 'session'